# Iris Flower Classification Using Machine Learning

Predict the species of an Iris flower (**Setosa, Versicolor, Virginica**) from four measurements.

**How to use this notebook:** run the cells from top to bottom (Runtime → Run all). Read the text above each cell first. No results are written by hand — every number comes from code that you run.

**Google Colab users:** the next cell asks you to upload `Iris.csv`. If you skip it, the notebook falls back to the copy of the dataset that ships with scikit-learn (same data).

## Step 0: Imports and settings
*What it does:* loads the libraries (pandas for tables, NumPy for numbers, Matplotlib/Seaborn for charts, scikit-learn for machine learning).
*Why:* Python only knows what you import. `RANDOM_STATE = 42` makes every random choice repeatable, so you get the same results as everyone else.
*Common error:* `ModuleNotFoundError` → run `!pip install pandas numpy matplotlib seaborn scikit-learn` in a cell, then re-run.

In [ ]:
# Colab only: upload Iris.csv (skipped automatically when not on Colab)
from pathlib import Path
try:
    from google.colab import files
    if not (Path("data/Iris.csv").exists() or Path("Iris.csv").exists()):
        print("Please choose Iris.csv from your computer (or cancel to use the built-in copy).")
        files.upload()
except ImportError:
    pass

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from matplotlib.colors import ListedColormap
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, f1_score,
)

RANDOM_STATE = 42            # makes results identical on every run
# The script looks for the CSV in these places (first match wins):
DATA_CANDIDATES = [Path("data/Iris.csv"), Path("Iris.csv")]
IMAGE_DIR = Path("images")
IMAGE_DIR.mkdir(exist_ok=True)

FEATURES = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
sns.set_theme(style="whitegrid")


def section(title):
    """Print a clear heading in the console."""
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)


def save_and_show(filename):
    """Save the current figure to /images and display it."""
    plt.tight_layout()
    plt.savefig(IMAGE_DIR / filename, dpi=150)
    plt.show()
    plt.close()

## Step 1: Load and inspect the dataset
*Dataset:* 150 flowers, 50 per species. *Features (inputs):* sepal length, sepal width, petal length, petal width, all in centimetres. *Target (what we predict):* the species.
*What the code does:* reads the CSV, renames columns to short names, removes the `Id` column (a row number carries no flower information) and removes the `Iris-` prefix from labels. Then it shows the first five rows, data types, missing values and duplicates.
*How to read the output:* "0 missing values" means no gaps to fix. A few duplicate rows are normal in this dataset (different flowers can have identical measurements) and are harmless here.
*Common error:* `FileNotFoundError` → make sure `Iris.csv` is in the same folder as the notebook (or inside a `data` folder).

In [ ]:
def load_data():
    """Load Iris.csv. If it is missing, fall back to scikit-learn's built-in copy."""
    data_path = next((p for p in DATA_CANDIDATES if p.exists()), None)
    if data_path is not None:
        df = pd.read_csv(data_path)
        print(f"Loaded dataset from: {data_path}")
        df = df.rename(columns={
            "SepalLengthCm": "sepal_length", "SepalWidthCm": "sepal_width",
            "PetalLengthCm": "petal_length", "PetalWidthCm": "petal_width",
            "Species": "species",
        })
        df = df.drop(columns=["Id"], errors="ignore")   # Id is just a row number
        df["species"] = df["species"].str.replace("Iris-", "", regex=False)  # 'Iris-setosa' -> 'setosa'
    else:
        print("Iris.csv not found -> using scikit-learn's built-in Iris dataset.")
        iris = load_iris(as_frame=True)
        df = iris.frame.rename(columns=dict(zip(iris.feature_names, FEATURES)))
        df["species"] = iris.target.map(dict(enumerate(iris.target_names)))
        df = df.drop(columns=["target"], errors="ignore")
    return df


def inspect_data(df):
    section("STEP 1: DATASET OVERVIEW")
    print(f"Samples (rows): {df.shape[0]}   Columns: {df.shape[1]}")
    print(f"Features: {FEATURES}")
    print(f"Target classes: {sorted(str(s) for s in df['species'].unique())}")
    print("\nFirst five rows:")
    print(df.head().to_string())
    print("\nData types / non-null counts:")
    df.info()
    print("\nMissing values per column:")
    print(df.isnull().sum().to_string())
    print(f"\nDuplicate rows: {df.duplicated().sum()}")
    print("\nSamples per species:")
    print(df["species"].value_counts().to_string())

In [ ]:
df = load_data()
inspect_data(df)

## Step 2: Exploratory Data Analysis (EDA)
*What it does:* summary statistics (mean, min, max…) and four kinds of charts: histograms, box plots, a pairplot and a correlation heatmap.
*Why:* before building a model you should *look* at the data. It tells you which measurements separate the species and whether the problem looks easy.
*Key functions:* `df.describe()` → statistics; `sns.histplot(hue=...)` → one colour per species; `sns.pairplot` → every feature against every other; `df.corr()` → how strongly two features move together (−1 to +1).

In [ ]:
def explore_data(df):
    section("STEP 2: EXPLORATORY DATA ANALYSIS")
    print("Summary statistics (all flowers):")
    print(df[FEATURES].describe().round(2).to_string())
    print("\nAverage measurements per species:")
    print(df.groupby("species")[FEATURES].mean().round(2).to_string())

    # Plot 1: histograms of each feature, coloured by species
    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    for ax, feat in zip(axes.ravel(), FEATURES):
        sns.histplot(data=df, x=feat, hue="species", kde=True, ax=ax)
        ax.set_title(f"Distribution of {feat.replace('_', ' ')}")
        ax.set_xlabel(f"{feat.replace('_', ' ')} (cm)")
        ax.set_ylabel("Number of flowers")
    save_and_show("01_feature_histograms.png")

    # Plot 2: box plots to compare species and spot outliers
    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    for ax, feat in zip(axes.ravel(), FEATURES):
        sns.boxplot(data=df, x="species", y=feat, hue="species", legend=False, ax=ax)
        ax.set_title(f"{feat.replace('_', ' ')} by species")
        ax.set_xlabel("Species")
        ax.set_ylabel(f"{feat.replace('_', ' ')} (cm)")
    save_and_show("02_boxplots.png")

    # Plot 3: pairplot = every feature against every other feature
    pair = sns.pairplot(df, hue="species", corner=True)
    pair.figure.suptitle("Pairplot of Iris Measurements", y=1.02)
    pair.figure.savefig(IMAGE_DIR / "03_pairplot.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")

    # Plot 4: correlation heatmap
    plt.figure(figsize=(6, 5))
    sns.heatmap(df[FEATURES].corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
    plt.title("Correlation Between Features")
    save_and_show("04_correlation_heatmap.png")

    print("\nWhat the plots tell us (plain English):")
    print("- Setosa has much smaller petals than the other two species and is easy to separate.")
    print("- Versicolor and Virginica overlap a lot in sepal size and only slightly in petal size.")
    print("- Petal length and petal width separate the species best.")
    print("- Petal length and petal width are strongly correlated (about 0.96): they grow together.")

In [ ]:
explore_data(df)

## Step 3: Preprocessing — features, target and train/test split
*What it does:* `X` holds the four measurements, `y` holds the species. `train_test_split` keeps 80% of flowers for learning and hides 20% for the final exam.
*Why separate them?* A student who sees the exam questions in advance can memorise the answers. The model is the same: only unseen data shows whether it really learned.
*Parameters:* `test_size=0.20` → 20% for testing; `random_state=42` → same split every run; `stratify=y` → each species appears in equal proportions in both parts (40/40/40 train and 10/10/10 test).

In [ ]:
def prepare_data(df):
    section("STEP 3: DATA PREPROCESSING")
    X = df[FEATURES]        # independent variables (inputs)
    y = df["species"]       # dependent variable (what we predict)

    # Why split? A model that is tested on data it already saw can just
    # memorise it. The test set is "unseen" data that shows how the model
    # behaves on new flowers.
    # stratify=y keeps the same species proportions in both sets.
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
    )
    print(f"Training samples: {len(X_train)} ({len(X_train) / len(df):.0%})")
    print(f"Testing samples : {len(X_test)} ({len(X_test) / len(df):.0%})")
    print("\nSpecies counts in training set:\n", y_train.value_counts().to_string())
    print("\nSpecies counts in testing set:\n", y_test.value_counts().to_string())
    return X_train, X_test, y_train, y_test

In [ ]:
X_train, X_test, y_train, y_test = prepare_data(df)

## Step 4: Train the models
*Main model:* **Logistic Regression** (despite its name, it is a classifier: it calculates a probability for each species and picks the highest).
*Comparison models:* Decision Tree and K-Nearest Neighbors.
*Pipeline:* `StandardScaler` rescales every feature to a similar range, then the model learns. Keeping both in one `Pipeline` means the scaler only learns from training data, so nothing leaks from the test set.
*Key lines:* `pipe.fit(X_train, y_train)` → learn; `pipe.predict(X_test)` → guess; `cross_val_score` → repeats training on 5 different slices of the training data as an extra reliability check.
*Reading the table:* compare "CV accuracy" (training data, 5-fold) with "Test accuracy". With only 30 test flowers, one flower changes accuracy by 3.3 percentage points, so small differences between models are not meaningful.

In [ ]:
def build_models():
    """Each model is a Pipeline: scale the numbers first, then classify.
    Putting the scaler inside the pipeline means it learns its mean/std
    from the training data only (prevents data leakage)."""
    return {
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=200, random_state=RANDOM_STATE)),
        ]),
        "Decision Tree": Pipeline([
            ("scaler", StandardScaler()),
            ("model", DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)),
        ]),
        "K-Nearest Neighbors": Pipeline([
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier(n_neighbors=5)),
        ]),
    }


def train_and_compare(models, X_train, X_test, y_train, y_test):
    section("STEP 4: MODEL TRAINING AND COMPARISON")
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    rows = []
    for name, pipe in models.items():
        pipe.fit(X_train, y_train)                                   # learn from training data
        test_acc = accuracy_score(y_test, pipe.predict(X_test))      # score on unseen data
        cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv)   # 5-fold check on training data
        rows.append({"Model": name,
                     "CV accuracy (train, mean)": round(cv_scores.mean(), 4),
                     "CV std": round(cv_scores.std(), 4),
                     "Test accuracy": round(test_acc, 4)})
    results = pd.DataFrame(rows)
    print(results.to_string(index=False))
    print("\nLogistic Regression is the main model required by the project;")
    print("the other two are included only for comparison.")
    return results

In [ ]:
models = build_models()
results = train_and_compare(models, X_train, X_test, y_train, y_test)
final_model = models["Logistic Regression"]

## Step 5: Evaluate Logistic Regression
Simple example with 10 virginica flowers where the model finds 9 and also mislabels 1 versicolor as virginica:
* **Accuracy** – share of all flowers named correctly.
* **Precision** (virginica) – of everything *predicted* virginica, how much really is? (9 of 10 → 0.90)
* **Recall** (virginica) – of all *real* virginica, how many were found? (9 of 10 → 0.90)
* **F1-score** – one number balancing precision and recall.
* **Confusion matrix** – rows are the real species, columns are the predictions; the diagonal holds correct answers.

`macro` averaging gives each species equal weight. All numbers below are computed from your test set.

In [ ]:
def evaluate_model(model, X_test, y_test):
    section("STEP 5: EVALUATION OF LOGISTIC REGRESSION")
    y_pred = model.predict(X_test)
    labels = list(model.classes_)       # class order used everywhere below

    acc = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {acc:.4f}  ({int(round(acc * len(y_test)))} of {len(y_test)} correct)")

    cm = confusion_matrix(y_test, y_pred, labels=labels)
    print("\nConfusion matrix (rows = actual, columns = predicted):")
    print(pd.DataFrame(cm, index=[f"actual {l}" for l in labels],
                       columns=[f"pred {l}" for l in labels]).to_string())

    print("\nPrecision / Recall / F1 (macro = simple average over the 3 classes):")
    print(f"Precision: {precision_score(y_test, y_pred, average='macro'):.4f}")
    print(f"Recall   : {recall_score(y_test, y_pred, average='macro'):.4f}")
    print(f"F1-score : {f1_score(y_test, y_pred, average='macro'):.4f}")

    print("\nFull classification report:")
    print(classification_report(y_test, y_pred, labels=labels, digits=3))

    print("How to read these metrics (simple example):")
    print("- Accuracy : share of all flowers named correctly. 9 right out of 10 -> 0.90.")
    print("- Precision: of flowers PREDICTED as 'virginica', how many really are virginica?")
    print("- Recall   : of flowers that ARE 'virginica', how many did we find?")
    print("- F1-score : one number balancing precision and recall (their harmonic mean).")
    print("- Confusion matrix: the diagonal = correct; anything off the diagonal = a mistake.")
    return y_pred, cm, labels

In [ ]:
y_pred, cm, labels = evaluate_model(final_model, X_test, y_test)

## Step 6: Visualizations
Confusion-matrix heatmap, two scatter plots (petal and sepal), a model-comparison bar chart and a decision-region plot that shows where the model draws the borders between species (drawn with only the two petal features so it fits on a flat chart).

In [ ]:
def plot_confusion_matrix(cm, labels):
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels)
    plt.title("Confusion Matrix - Logistic Regression (Test Set)")
    plt.xlabel("Predicted species")
    plt.ylabel("Actual species")
    save_and_show("05_confusion_matrix.png")


def plot_scatter(df):
    plt.figure(figsize=(7, 5))
    sns.scatterplot(data=df, x="petal_length", y="petal_width",
                    hue="species", style="species", s=70)
    plt.title("Petal Length vs Petal Width")
    plt.xlabel("Petal length (cm)")
    plt.ylabel("Petal width (cm)")
    plt.legend(title="Species")
    save_and_show("06_scatter_petal.png")

    plt.figure(figsize=(7, 5))
    sns.scatterplot(data=df, x="sepal_length", y="sepal_width",
                    hue="species", style="species", s=70)
    plt.title("Sepal Length vs Sepal Width")
    plt.xlabel("Sepal length (cm)")
    plt.ylabel("Sepal width (cm)")
    plt.legend(title="Species")
    save_and_show("07_scatter_sepal.png")


def plot_model_comparison(results):
    plt.figure(figsize=(7, 4.5))
    ax = sns.barplot(data=results, x="Model", y="Test accuracy", hue="Model", legend=False)
    for container in ax.containers:
        ax.bar_label(container, fmt="%.3f")
    plt.ylim(0, 1.1)
    plt.title("Test Accuracy by Model")
    plt.xlabel("Model")
    plt.ylabel("Accuracy")
    save_and_show("08_model_comparison.png")


def plot_decision_regions(df, X_train, y_train):
    """Extra visual: where the model draws boundaries between species.
    Uses only the 2 petal features so the result can be drawn in 2D
    (this small model is for illustration only, not the main model)."""
    cols = ["petal_length", "petal_width"]
    model2d = Pipeline([("scaler", StandardScaler()),
                        ("model", LogisticRegression(max_iter=200, random_state=RANDOM_STATE))])
    model2d.fit(X_train[cols], y_train)

    xx, yy = np.meshgrid(
        np.linspace(df[cols[0]].min() - 0.5, df[cols[0]].max() + 0.5, 300),
        np.linspace(df[cols[1]].min() - 0.5, df[cols[1]].max() + 0.5, 300))
    grid = pd.DataFrame(np.c_[xx.ravel(), yy.ravel()], columns=cols)
    class_idx = {c: i for i, c in enumerate(model2d.classes_)}
    zz = pd.Series(model2d.predict(grid)).map(class_idx).to_numpy().reshape(xx.shape)

    plt.figure(figsize=(7, 5))
    palette = sns.color_palette()[:3]   # same colours for background regions and points
    plt.contourf(xx, yy, zz, alpha=0.2, cmap=ListedColormap(palette), levels=np.arange(-0.5, 3, 1))
    sns.scatterplot(data=df, x=cols[0], y=cols[1], hue="species", style="species",
                    hue_order=list(model2d.classes_), palette=palette, s=60)
    plt.title("Decision Regions (Logistic Regression on Petal Features)")
    plt.xlabel("Petal length (cm)")
    plt.ylabel("Petal width (cm)")
    plt.legend(title="Species")
    save_and_show("09_decision_regions.png")

In [ ]:
plot_confusion_matrix(cm, labels)
plot_scatter(df)
plot_model_comparison(results)
plot_decision_regions(df, X_train, y_train)

## Step 7: Prediction system
`predict_species` first **validates** the input (must be a number, greater than 0 and at most 15 cm), puts it in a table with the same column names used in training, then returns the species and the model's confidence. Bad input raises a clear error message instead of crashing.

In [ ]:
def predict_species(model, sepal_length, sepal_width, petal_length, petal_width):
    """Validate four measurements (cm) and return the predicted species name."""
    values = {"sepal_length": sepal_length, "sepal_width": sepal_width,
              "petal_length": petal_length, "petal_width": petal_width}
    cleaned = []
    for name, value in values.items():
        try:
            number = float(value)
        except (TypeError, ValueError):
            raise ValueError(f"{name} must be a number, got {value!r}.")
        if not np.isfinite(number):
            raise ValueError(f"{name} must be a finite number.")
        if not 0 < number <= 15:     # generous limit; real Iris values are roughly 0.1-8 cm
            raise ValueError(f"{name} must be between 0 and 15 cm, got {number}.")
        cleaned.append(number)

    row = pd.DataFrame([cleaned], columns=FEATURES)   # same column names as training
    species = model.predict(row)[0]
    proba = model.predict_proba(row)[0]
    confidence = proba.max()
    return species, confidence


def show_prediction(model, *measurements):
    try:
        species, conf = predict_species(model, *measurements)
        print(f"Input {measurements} -> Predicted: Iris {species}  (confidence {conf:.1%})")
    except ValueError as err:
        print(f"Input {measurements} -> Invalid input: {err}")


def interactive_loop(model):
    print("\nType 4 measurements in cm, e.g. 5.1 3.5 1.4 0.2   (or 'q' to quit)")
    while True:
        text = input("sepal_length sepal_width petal_length petal_width > ").strip()
        if text.lower() in {"q", "quit", "exit"}:
            break
        parts = text.replace(",", " ").split()
        if len(parts) != 4:
            print("Please enter exactly 4 numbers.")
            continue
        show_prediction(model, *parts)

In [ ]:
show_prediction(final_model, 5.1, 3.5, 1.4, 0.2)    # looks like setosa
show_prediction(final_model, 6.0, 2.9, 4.5, 1.5)    # looks like versicolor
show_prediction(final_model, 6.9, 3.1, 5.8, 2.3)    # looks like virginica
show_prediction(final_model, -1, 3.0, 4.0, 1.0)     # invalid: negative
show_prediction(final_model, "abc", 3.0, 4.0, 1.0)  # invalid: not a number

### Try your own flower
Change the four numbers (centimetres) and run the cell.

In [ ]:
show_prediction(final_model, 5.8, 2.7, 5.1, 1.9)   # sepal_length, sepal_width, petal_length, petal_width

## Conclusion and limits
Logistic Regression separates the three species well on this dataset. Setosa is perfectly separable; most errors occur between Versicolor and Virginica. Limits: only 150 samples, a 30-flower test set (results vary with the split), and one clean, well-known dataset. This is a learning project and **not production-ready**.